# Graphene/Ni(111) Interface: Registry, Separation and Work of Adhesion

## 0. Introduction

This notebook reproduces the structure and energetics of graphene on Ni(111) following the review:

> **Arjun Dahal, Matthias Batzill**
> "Graphene–nickel interfaces: a review"
> Nanoscale, 6(5), 2548. (2014)
> [DOI: 10.1039/c3nr05279f](https://doi.org/10.1039/c3nr05279f)

The review's structural facts (its section 2.1): graphene locks into a 1×1 registry on Ni(111);
LEED I–V and ion scattering identify the adsorbed structure as one carbon **atop** a first-layer Ni
and the other in the **fcc hollow**, 0.211 nm above the surface with a 0.005 nm buckling in which
the atop carbon sits further out. Its computed numbers come from
[Lahiri et al., New J. Phys. 13, 025001 (2011)](https://doi.org/10.1088/1367-2630/13/2/025001)
(open access), whose Table 1 is the quantitative target here:

| interface | work of adhesion (J/m²) | separation (Å) |
|---|---|---|
| fcc (atop + fcc hollow) | 0.81 | 2.16 |
| hcp (atop + hcp hollow) | 0.77 | 2.17 |
| hollow (fcc + hcp hollows) | 0.31 | 3.26 |

(The review's text quotes the hollow as 0.38 J/m²; the source paper's Table 1 says 0.31 — this
notebook targets the source.) The four candidate registries, in the review's own Fig. 1:

<img src="https://github.com/Exabyte-io/documentation/raw/12617167278ae3523adc028583b21ea4e8ebd197/images/tutorials/materials/optimization/optimization_interface_film_xy_position_graphene_nickel/0-figure-from-manuscript.webp" alt="The four registries of graphene on a close-packed metal surface" width="600"/>

The bridge registry (d) is not quantified in either paper — it is included here as an extra point
beyond the published set.

The published calculation (Lahiri et al., section 2.2) used **LDA, spin-polarized, with geometry
relaxation** — five Ni layers with the bottom two fixed — because "GGA does not provide an adequate
description of Ni–graphene bonding". This notebook follows that recipe in two tiers:

- **Fast (here, in minutes):** each registry relaxed with the
  [MACE-MP](https://github.com/ACEsuit/mace) machine-learned force field (+D3), with the bottom
  substrate layers fixed as in the paper; same-cell references give the work of adhesion. MACE is
  PBE-trained, and PBE is exactly the functional the paper rejects for this system — so its
  chemisorption values are expected to underbind, and the notebook prints them **against** the
  paper's rather than pretending. The structure side (registry, separation trend, buckling sign,
  the hollow's dispersion-bound minimum) is where the fast tier earns its keep.
- **Precise (platform jobs):** the paper's functional — **LDA** (pz, ultrasoft), spin-polarized,
  **with relaxation**, no dispersion correction (LDA binds this interface unaided, which is why the
  paper chose it) — for each registry plus the two same-cell references the work of adhesion needs.

**Prerequisite:** run
[optimization_interface_film_xy_position_graphene_nickel.ipynb](optimization_interface_film_xy_position_graphene_nickel.ipynb)
first — it creates and saves the base interface material this notebook loads.

## 1. Prepare the Environment
### 1.1. Install Packages


In [ ]:
from mat3ra.notebooks_utils.mlff import get_mlff_install_profiles
from mat3ra.notebooks_utils.packages import install_packages

await install_packages(get_mlff_install_profiles("mace"))

from mat3ra.notebooks_utils.pyodide.packages.patches import apply_all_patches

apply_all_patches("mace")

### 1.2. Set Parameters


In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

# 2. Auth and organization parameters
ORGANIZATION_NAME = None

# 3. Material parameters
FOLDER = "./uploads"
BASE_MATERIAL_NAME = "Graphene_Nickel_interface"  # created by the companion structure notebook

# 4. MLFF parameters. MACE-MP-0 is trained on inorganic crystals and surfaces. The large model at
# float64 is not a preference: the medium model at float32 finds no chemisorbed minimum at all.
MACE_MODEL_FAMILY = "MACE-MP-0"
MACE_MODEL = "large"  # "small", "medium", "large"
MACE_DISPERSION = True  # D3; the hollow registry is dispersion-bound
MACE_DEFAULT_DTYPE = "float64"
MACE_DEVICE = "cpu"

# 5. Separation scan, in Angstrom — brackets the minima before relaxing. The window has to cover
# both published distances (2.16 A chemisorbed, 3.26 A for the hollow) with room on either side.
Z_SCAN_START = 1.8
Z_SCAN_STOP = 4.3
Z_SCAN_STEP = 0.25

# A chemisorbing registry has two minima: one where graphene bonds to the surface and one held
# only by dispersion, further out. Anything below 2.6 A is the chemisorbed branch by a wide
# margin either way (2.16 vs 3.26 A in the paper).
CHEMISORBED_BELOW = 2.6  # Angstrom

# 6. Relaxation — the paper's scheme: geometry optimization with the bottom substrate layers
# fixed. Relaxation is what produces the buckling, which is one of the published numbers.
FMAX = 0.02  # eV/A
FROZEN_SUBSTRATE_LAYERS = 2  # the paper fixes the bottom two of its five Ni layers

# 7. Workflow parameters
WORKFLOW_SEARCH_TERM = "total_energy.json"
APPLICATION_NAME = "espresso"
MY_WORKFLOW_NAME = "Total Energy (Gr/Ni registry)"

# Method parameters — the published setup where the platform can express it. Lahiri et al. used
# LDA, spin-polarized, with relaxation, and no dispersion correction: LDA binds this interface
# unaided, and that is the stated reason they chose it over GGA.
PSEUDOPOTENTIAL_TYPE = "us"  # GBRV ultrasoft; the platform carries the lda/pz set for Ni and C
FUNCTIONAL = "pz"  # LDA
MODEL_SUBTYPE = "lda"
ECUTWFC = 40   # GBRV publishes its ultrasoft set as a 40 / 200 Ry pair
ECUTRHO = 200

# K is at (1/3, 1/3), so in-plane divisions must be a multiple of three for the mesh to contain
# it, and a metal needs a dense mesh to resolve its Fermi surface.
SCF_KGRID = [12, 12, 1]

# Nickel is ferromagnetic — spin-polarized, started near its bulk moment (the paper's LDA value
# is 0.56 uB).
STARTING_MAGNETIZATION = {"Ni": 0.7}

# A spin-polarized metal slab is the hard case for SCF, and the platform defaults do not converge
# it: a first run stopped at "convergence NOT achieved after 100 iterations" with the total energy
# oscillating in its fourth decimal — charge sloshing, not divergence. Cold smearing, local-TF
# mixing and a smaller mixing fraction address exactly that.
SMEARING = "mv"  # Marzari-Vanderbilt cold smearing
DEGAUSS = 0.01  # Ry
ADDITIONAL_PARAMETERS = {
    "electrons": {
        "mixing_mode": "local-TF",
        "mixing_beta": 0.2,
        "electron_maxstep": 200,
    },
}

# 8. Compute parameters
CLUSTER_NAME = None
QUEUE_NAME = QueueName.D
PPN = 1

# 9. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 30


## 2. Load the Base Interface

The base interface is created by the companion structure notebook and saved into `uploads/`.
It is required — this notebook does not substitute another material.


In [ ]:
from mat3ra.made.material import Material
from mat3ra.made.tools.modify import interface_get_part
from mat3ra.made.tools.convert.interface_parts_enum import InterfacePartsEnum
from mat3ra.notebooks_utils.material import load_material_from_folder
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize

base_interface = load_material_from_folder(FOLDER, BASE_MATERIAL_NAME)
if base_interface is None:
    raise RuntimeError(
        f"'{BASE_MATERIAL_NAME}' not found in {FOLDER} — run "
        "optimization_interface_film_xy_position_graphene_nickel.ipynb first."
    )

film_part = interface_get_part(base_interface, part=InterfacePartsEnum.FILM)
substrate_part = interface_get_part(base_interface, part=InterfacePartsEnum.SUBSTRATE)

_cart = base_interface.clone()
_cart.to_cartesian()
film_cart = film_part.clone(); film_cart.to_cartesian()
substrate_cart = substrate_part.clone(); substrate_cart.to_cartesian()

film_z = [c[2] for c in film_cart.basis.coordinates.values]
substrate_z = [c[2] for c in substrate_cart.basis.coordinates.values]
measured_gap = min(film_z) - max(substrate_z)

print(f"Material:  {base_interface.name}")
from collections import Counter
composition = dict(Counter(base_interface.basis.elements.values))
print(f"Composition: {composition}")
print(f"Atoms:     {len(base_interface.basis.elements.values)} "
      f"({len(film_cart.basis.elements.values)} film C, {len(substrate_cart.basis.elements.values)} substrate Ni)")
print(f"Film-substrate plane distance as built: {measured_gap:.3f} A")

visualize([{"material": base_interface, "title": base_interface.name}], repetitions=[3, 3, 1], rotation="-90x")

## 3. Place the Film at the High-Symmetry Registries

The registries are defined by where carbon atoms sit relative to the Ni(111) surface sites:
**top** (above a first-layer Ni), **hcp hollow** (above a second-layer Ni), **fcc hollow**
(above a third-layer Ni), and **bridge** (midpoint of two neighboring first-layer Ni).
The sites are measured from the structure itself — the top three Ni layers — and the film is
translated so one carbon sublattice lands on each site in turn.


In [ ]:
import numpy as np

cell_2d = np.array(_cart.lattice.vector_arrays)[:2, :2]
c_xyz = np.array(film_cart.basis.coordinates.values)
ni_xyz = np.array(substrate_cart.basis.coordinates.values)
if len(c_xyz) != 2 or len(ni_xyz) < 3:
    raise RuntimeError("Expected the 1x1 interface: 2 carbons and >= 3 Ni layers")
c_a, c_b = c_xyz[0], c_xyz[1]

# In a 1x1 cell each Ni layer holds one atom, and the three surface sites project straight onto
# the top three layers: layer 1 = atop, layer 2 = hcp hollow, layer 3 = fcc hollow. This map is
# only for NAMING: the fcc and hcp hollows are identical from above and differ by what lies
# underneath, and the paper's numbers are per named registry.
ni_by_depth = ni_xyz[np.argsort(-ni_xyz[:, 2])]
site_xy = {"atop": ni_by_depth[0][:2], "hcp": ni_by_depth[1][:2], "fcc": ni_by_depth[2][:2]}

def site_of(point_xy):
    def distance(site):
        images = [site + i * cell_2d[0] + j * cell_2d[1] for i in (-1, 0, 1) for j in (-1, 0, 1)]
        return min(np.linalg.norm(im - point_xy) for im in images)
    named = {name: distance(s) for name, s in site_xy.items()}
    first, second = sorted(named.values())[:2]
    return None if second - first < 0.05 else min(named, key=named.get)  # None: refuse a tie

# The three stackings (review Fig. 1 a-c) are one site-to-site step apart: shifting the whole film
# by the atop-to-hcp vector moves every carbon one step along the atop -> hcp -> fcc cycle, so the
# shifts are 0, one step, two steps. In the bridge registry (d) the C-C bond straddles a
# first-layer Ni, which pins the bond midpoint over the atop site.
images = [site_xy['hcp'] + i * cell_2d[0] + j * cell_2d[1] for i in (-1, 0, 1) for j in (-1, 0, 1)]
site_step = min(images, key=lambda im: np.linalg.norm(im - site_xy['atop'])) - site_xy['atop']
displacements = {}
for n in (0, 1, 2):
    shift = n * site_step
    pair = {site_of(c_a[:2] + shift), site_of(c_b[:2] + shift)}
    label = "hollow" if pair == {"fcc", "hcp"} else f"atop_{(pair - {'atop'}).pop()}"
    displacements[label] = np.array([*shift, 0.0])
displacements["bridge"] = np.array([*(site_xy["atop"] - (c_a[:2] + c_b[:2]) / 2), 0.0])

if set(displacements) != {"hollow", "atop_fcc", "atop_hcp", "bridge"}:
    raise RuntimeError(f"Registry derivation produced {set(displacements)}")
for label, panel in (("hollow", "(a)"), ("atop_fcc", "(b)"), ("atop_hcp", "(c)"), ("bridge", "(d)")):
    print(f"{label:<10} Fig. 1 {panel}   film shift (A): {np.round(displacements[label][:2], 3)}")


In [ ]:
from mat3ra.made.tools.modify import interface_displace_part

def film_at(registry_label, plane_distance):
    displacement = displacements[registry_label] + np.array([0.0, 0.0, plane_distance - measured_gap])
    return interface_displace_part(base_interface, displacement=list(displacement))

preview = []
for label in displacements:
    m = film_at(label, measured_gap)
    m.name = f"{BASE_MATERIAL_NAME} {label}"
    preview.append({"material": m, "title": label})

visualize(preview, repetitions=[2, 2, 1])

## 4. Fast Tier: Relax Each Registry with MACE

Each registry is bracketed by a rigid scan, then **relaxed** — all atoms free, the bottom
substrate layers fixed, the paper's scheme — and the same-cell references (bare Ni slab,
free-standing graphene) are relaxed the same way, which turns total energies into a work of
adhesion: W = (E_slab + E_graphene − E_interface) / A. After each relaxation the registry is
re-measured from the final positions, so a structure that slid into a neighbouring registry
cannot be reported under the wrong name. Distances follow the paper's convention: the averaged
carbon height above the averaged top-Ni height; buckling is the height difference between the
two carbons, positive when the atop carbon sits further out.


In [ ]:
import importlib.util

from ase.constraints import FixAtoms
from ase.optimize import BFGS
from mat3ra.made.tools.convert import from_ase, to_ase
from mat3ra.notebooks_utils.mlff import create_mlff_calculator

# D3 needs the torch-dftd package. Where it is unavailable (the in-browser environment does not
# bundle it), MACE runs at plain PBE level — which is exactly the description the review rejects
# for this interface: chemisorption comes out unbound and the hollow registry loses its
# dispersion-bound minimum. The notebook states which picture it is computing.
dispersion_available = importlib.util.find_spec("torch_dftd") is not None
dispersion_active = MACE_DISPERSION and dispersion_available
if MACE_DISPERSION and not dispersion_available:
    print("torch-dftd is not available here: the fast tier runs WITHOUT dispersion — the")
    print("GGA-level picture the manuscript describes as inadequate for this interface.")

calculator = create_mlff_calculator(
    "mace",
    {
        "family": MACE_MODEL_FAMILY,
        "model": MACE_MODEL,
        "dispersion": dispersion_active,
        "default_dtype": MACE_DEFAULT_DTYPE,
        "device": MACE_DEVICE,
    },
)


In [ ]:
from ase.constraints import FixAtoms
from ase.optimize import BFGS
from mat3ra.made.tools.convert import from_ase, to_ase

EV_PER_A2_TO_J_PER_M2 = 16.0217663
n_carbon = len(film_cart.basis.elements.values)
film_elements = set(film_cart.basis.elements.values)
substrate_elements = set(substrate_cart.basis.elements.values)

def relax(atoms):
    """The paper's scheme: everything free except the bottom substrate layers."""
    z = atoms.positions[:, 2]
    substrate = [i for i, s in enumerate(atoms.get_chemical_symbols()) if s in substrate_elements]
    held = sorted(substrate, key=lambda i: z[i])[:FROZEN_SUBSTRATE_LAYERS]
    if held:
        atoms.set_constraint(FixAtoms(indices=held))
    atoms.calc = calculator
    BFGS(atoms).run(fmax=FMAX, steps=300)
    return atoms

def interface_geometry(atoms):
    """Separation to the top-Ni plane (the paper's convention: averaged carbon height);
    buckling signed positive when the atop carbon sits further out."""
    symbols, pos = atoms.get_chemical_symbols(), atoms.positions
    carbon = [i for i, s in enumerate(symbols) if s in film_elements]
    top_ni = max(pos[i, 2] for i, s in enumerate(symbols) if s in substrate_elements)
    separation = float(np.mean([pos[i, 2] for i in carbon]) - top_ni)
    atop = next((i for i in carbon if site_of(pos[i, :2]) == "atop"), None)
    if atop is None:
        return separation, float(abs(pos[carbon[0], 2] - pos[carbon[1], 2]))
    other = next(i for i in carbon if i != atop)
    return separation, float(pos[atop, 2] - pos[other, 2])

# Same-cell references, relaxed the same way, turn total energies into a work of adhesion.
slab_atoms = relax(to_ase(substrate_part))
sheet_atoms = to_ase(film_part)
sheet_atoms.calc = calculator
BFGS(sheet_atoms).run(fmax=FMAX, steps=300)
E_separated = float(slab_atoms.get_potential_energy()) + float(sheet_atoms.get_potential_energy())
cell = np.array(to_ase(base_interface).cell)
area = float(np.linalg.norm(np.cross(cell[0], cell[1])))


In [ ]:
distances = np.arange(Z_SCAN_START, Z_SCAN_STOP + 1e-9, Z_SCAN_STEP)

scan_results = {}
for label in displacements:
    energies = []
    for d in distances:
        atoms = to_ase(film_at(label, float(d)))
        atoms.calc = calculator
        energies.append(float(atoms.get_potential_energy()))
    energies = np.array(energies)

    # bracketed minimum per branch: the lowest scanned point that is not a window edge
    starts = {}
    for branch, in_branch in (("chem", distances < CHEMISORBED_BELOW), ("phys", distances >= CHEMISORBED_BELOW)):
        i = int(np.where(in_branch)[0][np.argmin(energies[in_branch])])
        if 0 < i < len(distances) - 1 and energies[i] <= min(energies[i - 1], energies[i + 1]):
            starts[branch] = float(distances[i])
    if not starts:
        scan_results[label] = {"energies": energies, "chem": None, "relaxed": None}
        print(f"{label:<10} unbound in this window" + ("" if dispersion_active else " (dispersion inactive)"))
        continue

    atoms = relax(to_ase(film_at(label, starts.get("chem", starts.get("phys")))))
    separation, buckling = interface_geometry(atoms)
    carbon_sites = {site_of(atoms.positions[i, :2]) for i, s in enumerate(atoms.get_chemical_symbols())
                    if s in film_elements}
    if label in ("atop_fcc", "atop_hcp", "hollow") and carbon_sites != set(label.replace("hollow", "fcc_hcp").split("_")):
        print(f"! {label}: relaxed onto {carbon_sites} — treat this row with suspicion")
    w_adh = (E_separated - float(atoms.get_potential_energy())) / area * EV_PER_A2_TO_J_PER_M2
    scan_results[label] = {"energies": energies, "chem": starts.get("chem"),
                           "relaxed": {"w_adh": w_adh, "separation": separation, "buckling": buckling,
                                       "material": Material.create(from_ase(atoms))}}
    print(f"{label:<10} relaxed: d = {separation:5.2f} A   buckling = {buckling:+.3f} A   W_adh = {w_adh:.2f} J/m^2")


In [ ]:
import plotly.graph_objects as go

reference = min(float(r["energies"].min()) for r in scan_results.values())
fig = go.Figure()
for label, r in scan_results.items():
    fig.add_trace(go.Scatter(x=distances, y=(r["energies"] - reference) * 1000 / n_carbon,
                             mode="lines+markers", name=label))
fig.update_layout(
    title="Rigid-scan energy vs. separation (bracketing only; the table below is relaxed)",
    xaxis_title="plane distance (A)",
    yaxis_title="energy above the deepest scanned point (meV / C atom)",
)
fig.show()


In [ ]:
# Lahiri et al. (2011) Table 1; the review's text quotes the hollow as 0.38 — the table says 0.31
PAPER = {"atop_fcc": (0.81, 2.16), "atop_hcp": (0.77, 2.17), "hollow": (0.31, 3.26)}

rows = {label: r["relaxed"] for label, r in scan_results.items() if r["relaxed"]}
print(f"{'registry':<10}{'W_adh':>7}{'paper':>7}     {'d':>5}{'paper':>7}    buckling")
for label, r in sorted(rows.items(), key=lambda kv: -kv[1]["w_adh"]):
    w, d = PAPER.get(label, ("—", "—"))
    print(f"{label:<10}{r['w_adh']:>7.2f}{w:>7}     {r['separation']:>5.2f}{d:>7}    {r['buckling']:+.3f}")
for label in set(scan_results) - set(rows):
    print(f"{label:<10}unbound here — paper: {PAPER[label][0]} J/m^2 at {PAPER[label][1]} A")

mace_reproduces = (
    all(label in rows for label in PAPER)
    and rows["atop_fcc"]["w_adh"] > rows["atop_hcp"]["w_adh"] > rows["hollow"]["w_adh"]
    and abs(rows["atop_fcc"]["w_adh"] - PAPER["atop_fcc"][0]) <= 0.15
    and abs(rows["atop_fcc"]["separation"] - PAPER["atop_fcc"][1]) <= 0.10
    and rows["atop_fcc"]["buckling"] > 0
)
print(f"\nReproduces Lahiri et al. Table 1 [MACE tier]: {'yes' if mace_reproduces else 'no'}")
print("(MACE is PBE-grade — the functional the paper rejects for this interface. "
      "The LDA tier below carries the reproduction claim.)")


## 5. Precise Tier: the Paper's LDA, Relaxed, on the Platform

One relaxation + total-energy job per selected registry, starting from the MACE-relaxed geometry,
plus the two same-cell references the work of adhesion needs — the paper's functional (LDA),
spin-polarized, no dispersion correction. A default run selects one registry (three jobs). An
**empty** list skips the platform tier entirely, which is what the automated test does: with
relaxation these jobs take longer than a browser test may wait.


In [ ]:
DFT_REGISTRY_NAMES = [
    "atop_fcc",
    # "atop_hcp",
    # "hollow",
    # "bridge",
]


In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"Using project: {projects[0]['name']} ({project_id})")

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

def submitted_copy(material, name):
    """QE needs ATOMIC_SPECIES and ATOMIC_POSITIONS to agree, and the film/substrate labels only
    served the displacement, so they are dropped from anything submitted."""
    m = material.clone()
    m.basis.labels.values = []
    m.name = name
    return Material.create(get_or_create_material(client, m, ACCOUNT_ID))

dft_materials, reference_materials = {}, {}
if DFT_REGISTRY_NAMES:
    for label in DFT_REGISTRY_NAMES:
        relaxed = scan_results[label]["relaxed"]
        saved = submitted_copy(relaxed["material"],
                               f"{BASE_MATERIAL_NAME} {label} d{relaxed['separation']:.2f} relaxed")
        dft_materials[label] = saved
        print(f"{label:<16} -> '{saved.name}' ({len(saved.basis.elements.values)} atoms)")
    # The references live in the same cell and run with the same settings, so the cell- and
    # sampling-dependent part of the error drops out of the work-of-adhesion difference.
    for name, part in (("substrate", substrate_part), ("film", film_part)):
        saved = submitted_copy(part, f"{BASE_MATERIAL_NAME} {name} reference")
        reference_materials[name] = saved
        print(f"{name + ' ref':<16} -> '{saved.name}' ({len(saved.basis.elements.values)} atoms)")
else:
    print("DFT tier skipped: no registries selected.")


In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.ade.application import Application

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)
print(f"Using application: {app.name}")

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow

workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(WORKFLOW_SEARCH_TERM)
workflow = Workflow.create(workflow_config)
workflow.name = MY_WORKFLOW_NAME

visualize_workflow(workflow)

In [ ]:
from mat3ra.mode import ModelFactory
from mat3ra.standata.model_tree import ModelTreeStandata

# The paper's functional: LDA describes this interface's geometry in agreement with experiment,
# which is the stated reason Lahiri et al. chose it over GGA. No dispersion correction on top.
model_config = ModelTreeStandata.get_model_by_parameters(
    type="dft",
    subtype=MODEL_SUBTYPE,
    functional=FUNCTIONAL,
)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = ModelFactory.create(model_config)


In [ ]:
from mat3ra.notebooks_utils.workflow import apply_scf_kgrid, patch_workflow_qe_input
from mat3ra.wode.context.providers import PlanewaveCutoffsContextProvider

def configure(built, with_ni_moment):
    """The published settings, on both the relaxation and the SCF unit."""
    built.add_relaxation()
    cutoffs = PlanewaveCutoffsContextProvider(wavefunction=ECUTWFC, density=ECUTRHO,
                                              isEdited=True).get_context_item_data()
    for subworkflow in built.subworkflows:
        subworkflow.model = model
        for unit in subworkflow.units:
            unit.add_context(cutoffs)
            subworkflow.set_unit(unit)
    for unit_name in ("pw_relax", "pw_scf"):
        apply_scf_kgrid(built, SCF_KGRID, material=reference_material, unit_name=unit_name)
    system = {"nspin": 2, "degauss": DEGAUSS, "smearing": SMEARING}
    if with_ni_moment:
        system["starting_magnetization(1)"] = STARTING_MAGNETIZATION["Ni"]
    patch_workflow_qe_input(built, {"system": system, **ADDITIONAL_PARAMETERS}, unit_names=["pw_relax", "pw_scf"])
    return built

if dft_materials:
    reference_material = dft_materials[DFT_REGISTRY_NAMES[0]]
    # Ni is species 1 in the interface and in the bare slab, so one magnetized workflow serves
    # both; the graphene reference has no Ni and gets its own, without the moment.
    if reference_material.basis.elements.values[0] != "Ni":
        raise RuntimeError("Expected Ni as the first species — the magnetization index assumes it")
    configure(workflow, with_ni_moment=True)


In [ ]:
from mat3ra.notebooks_utils.core.entity.workflow.api import get_or_create_workflow

saved_workflows = {}
if dft_materials:
    workflows = {"interface": workflow, "substrate": workflow}
    if "film" in reference_materials:
        film_workflow = Workflow.create(WorkflowStandata.filter_by_application(app.name)
                                        .get_by_name_first_match(WORKFLOW_SEARCH_TERM))
        film_workflow.name = f"{MY_WORKFLOW_NAME} film"
        workflows["film"] = configure(film_workflow, with_ni_moment=False)
    for key, built in workflows.items():
        saved_workflows[key] = Workflow.create(get_or_create_workflow(client, built, ACCOUNT_ID))
        print(f"{key:<12} -> workflow {saved_workflows[key].id}")


In [ ]:
clusters = client.clusters.list() if dft_materials else []
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

In [ ]:
from mat3ra.ide.compute import Compute

compute = None
if dft_materials:
    if CLUSTER_NAME:
        cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
    else:
        cluster = clusters[0]
    compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN)
    print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")


In [ ]:
from mat3ra.utils.namespace import dict_to_namespace_recursive
from mat3ra.notebooks_utils.job import create_job

def submit_job_for(label, saved_material, which="interface"):
    job_response = create_job(
        api_client=client,
        materials=[saved_material],
        workflow=workflows[which],
        project_id=project_id,
        owner_id=ACCOUNT_ID,
        prefix=f"{MY_WORKFLOW_NAME} {label} {timestamp}",
        compute=compute.to_dict(),
    )
    job_id = dict_to_namespace_recursive(job_response)._id
    print(f"{label:<16} -> job {job_id}")
    return job_id

jobs, reference_jobs = {}, {}
if dft_materials:
    jobs = {label: submit_job_for(label, m) for label, m in dft_materials.items()}
    reference_jobs = {name: submit_job_for(f"{name} reference", m, which=name)
                      for name, m in reference_materials.items()}


In [ ]:
for label, job_id in {**jobs, **reference_jobs}.items():
    client.jobs.submit(job_id)
    print(f"Submitted {label}: {job_id}")


In [ ]:
from mat3ra.notebooks_utils.api.job import wait_for_jobs_to_finish_async

all_job_ids = list(jobs.values()) + list(reference_jobs.values())
if all_job_ids:
    await wait_for_jobs_to_finish_async(client.jobs, all_job_ids, poll_interval=POLL_INTERVAL)
else:
    print("Nothing to wait for — the DFT tier was skipped.")


In [ ]:
from mat3ra.prode import PropertyName

dft_energies, reference_energies, dft_w_adh = {}, {}, {}
if jobs:
    def total_energy_of(job_id):
        property_data = client.properties.get_for_job(job_id, property_name=PropertyName.scalar.total_energy.value)
        return float(property_data[0]["data"]["value"])

    dft_energies = {label: total_energy_of(job_id) for label, job_id in jobs.items()}
    reference_energies = {name: total_energy_of(job_id) for name, job_id in reference_jobs.items()}
    separated = reference_energies["substrate"] + reference_energies["film"]
    dft_w_adh = {label: (separated - e) / area * EV_PER_A2_TO_J_PER_M2 for label, e in dft_energies.items()}

    print(f"{'registry':<12}{'E_DFT (eV)':<16}{'W_adh (J/m^2)':<15}{'paper (J/m^2)'}")
    for label, e in sorted(dft_energies.items(), key=lambda kv: kv[1]):
        print(f"{label:<12}{e:<16.4f}{dft_w_adh[label]:<15.2f}{PAPER.get(label, ('—',))[0]}")


## 6. Compare with the Article


In [ ]:
# The verdict, per tier, against Lahiri et al. (2011) Table 1 — reached through the review.
print("Targets: fcc 0.81 J/m^2 @ 2.16 A · hcp 0.77 @ 2.17 · hollow 0.31 @ 3.26 · "
      "buckling ~0.03 A, atop carbon out\n")
print(f"Reproduces Lahiri et al. Table 1 [MACE tier]: {'yes' if mace_reproduces else 'no'}")

dft_w_adh = globals().get("dft_w_adh", {})
if dft_w_adh:
    complete = all(label in dft_w_adh for label in PAPER)
    within = all(abs(dft_w_adh[label] - PAPER[label][0]) <= 0.15 for label in PAPER if label in dft_w_adh)
    ordered = (not complete) or (dft_w_adh["atop_fcc"] > dft_w_adh["atop_hcp"] > dft_w_adh["hollow"])
    for label in PAPER:
        if label in dft_w_adh:
            print(f"  {label:<10} W_adh = {dft_w_adh[label]:.2f} J/m^2   paper: {PAPER[label][0]}")
    suffix = "" if complete else f" ({len(dft_w_adh)} of {len(PAPER)} registries)"
    print(f"Reproduces Lahiri et al. Table 1 [DFT tier]: {'yes' if within and ordered and complete else 'no'}{suffix}")
else:
    print("DFT tier: not run — select registries in DFT_REGISTRY_NAMES for the paper's-functional verdict.")


## References

[1] Arjun Dahal, Matthias Batzill, "Graphene-nickel interfaces: a review",
Nanoscale 6(5), 2548 (2014). [DOI: 10.1039/c3nr05279f](https://doi.org/10.1039/c3nr05279f)

[2] Jayeeta Lahiri, Travis S. Miller, Andrew J. Ross, Lyudmyla Adamska, Ivan I. Oleynik,
Matthias Batzill, "Graphene growth and stability at nickel surfaces", New J. Phys. 13, 025001
(2011). [DOI: 10.1088/1367-2630/13/2/025001](https://doi.org/10.1088/1367-2630/13/2/025001)

[3] mat3ra-made: https://github.com/Exabyte-io/made

[4] MACE-MP-0 foundation models: https://github.com/ACEsuit/mace
